In [4]:
# Flickr-Geo suitability test (self-contained, schema-tolerant)
import duckdb, pandas as pd
from huggingface_hub import list_repo_files

REPO = "do-me/Flickr-Geo"
files = sorted(f for f in list_repo_files(REPO, repo_type="dataset") if f.endswith(".parquet"))
print("parquet files:", len(files), files[:2])

N = 8
step = max(len(files) // N, 1)
sample = files[::step][:N]
urls = [f"hf://datasets/{REPO}/{f}" for f in sample]
SRC = f"read_parquet({urls!r}, union_by_name=true, filename=true)"

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")

# STEP 0: per-file health (rows, rows with coordinates, rows with text)
print(con.sql(f"""
  SELECT regexp_extract(filename, '[^/]+$') AS file,
         count(*) AS rows,
         count(TRY_CAST(latitude AS DOUBLE)) AS with_lat,
         count(*) FILTER (WHERE length(trim(COALESCE(CAST(description AS VARCHAR), ''))) > 0) AS with_descr,
         count(*) FILTER (WHERE length(trim(COALESCE(CAST(title AS VARCHAR), ''))) > 0) AS with_title
  FROM {SRC} GROUP BY 1 ORDER BY 1
""").df())

# STEP 1: unified schema
schema = con.sql(f"DESCRIBE SELECT * FROM {SRC}").df()
print(schema[["column_name", "column_type"]].to_string())
cols = set(schema["column_name"])
assert {"title", "description", "latitude", "longitude"} <= cols, "missing text/geo columns"
has_acc, has_owner = "accuracy" in cols, "owner" in cols

# STEP 2: quality statistics (only rows with valid coordinates)
generic = r"^(img|dsc|dscn|dscf|dsf|pxl|mvi|gopr|dji|p\d|_mg|image|photo|screenshot)[_\- ]?\d*"
base = f"""
  SELECT
    {"CAST(owner AS VARCHAR)" if has_owner else "NULL::VARCHAR"} AS owner,
    TRY_CAST(latitude AS DOUBLE)  AS lat,
    TRY_CAST(longitude AS DOUBLE) AS lon,
    {"TRY_CAST(accuracy AS INT)" if has_acc else "NULL::INT"} AS acc,
    COALESCE(CAST(title AS VARCHAR), '') AS title,
    COALESCE(CAST(description AS VARCHAR), '') AS descr,
    COALESCE(CAST(tags AS VARCHAR), '') AS tags
  FROM {SRC}
"""
con.execute(f"CREATE OR REPLACE TEMP VIEW t AS SELECT * FROM ({base}) WHERE lat IS NOT NULL AND lon IS NOT NULL AND NOT (lat = 0 AND lon = 0)")
con.execute(f"""
  CREATE OR REPLACE TEMP VIEW tt AS SELECT *,
    (len(string_split(trim(descr), ' ')) >= 5 AND length(trim(descr)) > 0) AS descr_5w,
    (NOT regexp_matches(lower(title), '{generic}') AND length(trim(title)) >= 15) AS title_ok
  FROM t""")
print(con.sql("""
  SELECT count(*) AS rows,
         count(DISTINCT (round(lat, 5), round(lon, 5))) AS distinct_coords,
         avg((acc >= 15)::INT) AS share_acc_ge15,
         avg((length(trim(descr)) > 0)::INT) AS share_descr,
         avg(descr_5w::INT) AS share_descr_5w,
         avg(title_ok::INT) AS share_title_nongeneric,
         avg((length(trim(tags)) > 0)::INT) AS share_tags,
         avg((descr_5w OR title_ok)::INT) AS share_usable_text
  FROM tt""").df().T)

# STEP 3 + 4 (fixed): estimated totals per city and owner concentration
cities = {
  "London": (51.28, 51.70, -0.51, 0.33), "New York": (40.49, 40.92, -74.26, -73.70),
  "Paris": (48.80, 48.92, 2.22, 2.47), "San Francisco": (37.70, 37.83, -122.52, -122.35),
  "Tokyo": (35.53, 35.82, 139.55, 139.93), "Berlin": (52.34, 52.68, 13.09, 13.76),
}
scale = len(files) / len(sample)
rows = []
for name, (a, b, c, d) in cities.items():
    n, u = con.execute(f"""SELECT count(*), avg((descr_5w OR title_ok)::INT) FROM tt
                           WHERE lat BETWEEN {a} AND {b} AND lon BETWEEN {c} AND {d}""").fetchone()
    rows.append((name, n, round(n * scale), None if u is None else round(u, 3)))
print(pd.DataFrame(rows, columns=["city", "rows_in_sample", "est_total_rows", "share_usable_text"]))

print(con.execute("""
  WITH ldn AS (SELECT owner, count(*) n FROM tt
               WHERE lat BETWEEN 51.28 AND 51.70 AND lon BETWEEN -0.51 AND 0.33 GROUP BY owner)
  SELECT sum(n) FILTER (WHERE rk <= 10) / sum(n) AS top10_owner_share, count(*) AS owners
  FROM (SELECT n, row_number() OVER (ORDER BY n DESC) rk FROM ldn)""").df())

parquet files: 5150 ['part-00000.parquet', 'part-00001.parquet']
                 file   rows  with_lat  with_descr  with_title
0  part-00643.parquet   9029      9029        5247        8893
1  part-01286.parquet  43042     43042       14500       40168
2  part-01929.parquet  48124     48124       16450       44578
3  part-02572.parquet  45094     45094       16011       42209
4  part-03215.parquet  46310     46310       15686       43020
5  part-03858.parquet  47065     47065       15434       43805
6  part-04501.parquet   4163      4163        1378        4127
       column_name column_type
0               id     VARCHAR
1            owner     VARCHAR
2           url_sq     VARCHAR
3         width_sq     INTEGER
4        height_sq     INTEGER
5            url_q     VARCHAR
6          width_q     INTEGER
7         height_q     INTEGER
8            url_t     VARCHAR
9          width_t     INTEGER
10        height_t     INTEGER
11           url_s     VARCHAR
12         width_s      DOUB

In [6]:
# London bbox checks: years, repeated captions, top titles, coordinate/owner concentration, non-Latin share
LDN = "lat BETWEEN 51.28 AND 51.70 AND lon BETWEEN -0.51 AND 0.33 AND acc >= 15"

con.execute(f"""CREATE OR REPLACE TEMP VIEW ldn AS
  SELECT TRY_CAST(latitude AS DOUBLE) lat, TRY_CAST(longitude AS DOUBLE) lon,
         TRY_CAST(accuracy AS INT) acc, CAST(owner AS VARCHAR) owner,
         COALESCE(CAST(title AS VARCHAR),'') title, COALESCE(CAST(description AS VARCHAR),'') descr,
         COALESCE(CAST(tags AS VARCHAR),'') tags, CAST(datetaken AS VARCHAR) dt, license
  FROM {SRC}""").fetchall()

print("London rows (acc>=15) in sample:", con.execute(f"SELECT count(*) FROM ldn WHERE {LDN}").fetchall()[0][0])

print(con.execute(f"""SELECT substr(dt,1,4) AS year, count(*) n FROM ldn WHERE {LDN}
                      GROUP BY 1 ORDER BY 1""").df())

# most repeated titles (landmarks / bulk captions)
print(con.execute(f"""SELECT lower(title) t, count(*) n FROM ldn WHERE {LDN} AND length(trim(title)) > 0
                      GROUP BY 1 ORDER BY n DESC LIMIT 15""").df())

# most repeated descriptions (bulk uploads)
print(con.execute(f"""SELECT left(lower(descr), 60) d, count(*) n FROM ldn WHERE {LDN} AND length(trim(descr)) > 0
                      GROUP BY 1 ORDER BY n DESC LIMIT 10""").df())

# coordinate concentration and non-ASCII share (rough language signal)
print(con.execute(f"""SELECT count(*) n, count(DISTINCT (round(lat,5), round(lon,5))) coords,
                             avg(regexp_matches(title || descr, '[^\\x00-\\x7F]')::INT) AS share_non_ascii
                      FROM ldn WHERE {LDN}""").df())
print(con.execute(f"""SELECT round(lat,4) la, round(lon,4) lo, count(*) n FROM ldn WHERE {LDN}
                      GROUP BY 1,2 ORDER BY n DESC LIMIT 10""").df())

London rows (acc>=15) in sample: 4356
    year    n
0   1926    1
1   1968    1
2   1969    1
3   1970    1
4   1971    2
5   1977    2
6   1981    1
7   1982    1
8   1986    2
9   1987    2
10  1989    2
11  1990    3
12  1992    1
13  2000    7
14  2001    4
15  2002   62
16  2003    6
17  2004   34
18  2005   35
19  2006  259
20  2007  114
21  2008  266
22  2009  289
23  2010  266
24  2011  218
25  2012  360
26  2013  326
27  2014  222
28  2015  198
29  2016  177
30  2017  250
31  2018  217
32  2019  257
33  2020  105
34  2021  218
35  2022  241
36  2023  178
37  2024   27
                                                    t   n
0                               london september 2021  87
1                         help the hospices santa run  51
2   london, england, uk - the regent's park - quee...  49
3                                   opposites attract  23
4                                              london  19
5                             british 10k london 2009  18
6         